# 02 — Train Tier 1: QLoRA SFT

**Runs on:** Google Colab Free (T4 15 GB).

## How to use this notebook
1. Open in Colab. Set runtime → GPU (T4).
2. (Optional) Edit the `MODEL_NAME` in the **CONFIG** cell to the winner from `02a_model_evaluation.ipynb`. Default is Phi-4-mini.
3. **Runtime → Run all.**
4. When prompted, upload `train.jsonl`, `val.jsonl` (from `tools/yen_sei/data/refined/`). Upload any `test_*.jsonl` + `test_*_metadata.jsonl` sidecar files for evaluation.
5. Wait ~3–6 hours. The notebook will:
   - QLoRA fine-tune the chosen model (3 epochs, early stop on val loss).
   - Run inference on the named test sets + report format compliance and quality metrics.
   - Save adapter, merged-fp16 weights (if VRAM allows), and zip them for download.

**Total user effort: pick the runtime, click Run all, drop the data files. No copy-paste of code.**

In [ ]:
# =====================================================================
# Cell 1 — Install dependencies (silent). ~2-3 min on first run.
# =====================================================================
import subprocess, sys, os, time

PKGS = [
    "unsloth",
    "peft>=0.11.0",
    "transformers>=4.45.0",
    "datasets>=2.20.0",
    "bitsandbytes>=0.43.0",
    "accelerate>=0.34.0",
    "trl>=0.10.0",
]
print("Installing dependencies (silent)…")
t0 = time.time()
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U", *PKGS],
    check=True,
)
print(f"Done in {time.time() - t0:.1f}s.")


In [ ]:
# =====================================================================
# Cell 1.5 - Bundle tools.yen_sei.eval into Colab.
#
# The notebook imports `from tools.yen_sei.eval.runner import evaluate_test_sets`
# but Colab does not have the yen-go repo. This cell writes the 4 small
# files (~22 KB total) that the eval runner needs into /content/yen_sei_bundle/
# and adds it to sys.path. No repo clone, no extra upload required.
#
# To regenerate after editing runner/scorers/judges:
#   python tools/yen_sei/notebooks/_gen_bundle_cell.py
# then paste the output back into both notebooks.
# =====================================================================
import sys
from pathlib import Path

BUNDLE_ROOT = Path("/content/yen_sei_bundle")

_GO_CONST = r'''"""Colab stub: only GO_TECHNIQUE_PATTERN is used by scorers.py."""
import re

GO_TECHNIQUES = frozenset({
    "net", "geta", "ladder", "shicho", "snapback", "ko", "seki",
    "tesuji", "life", "death", "kill", "capture", "connect", "cut",
    "eye", "liberty", "liberties", "atari", "shortage", "semeai",
    "throw-in", "squeeze", "placement", "hane", "descent", "clamp",
    "wedge", "peep", "bamboo", "tiger", "empty triangle", "ponnuki",
    "alive", "dead", "unconditionally", "approach", "invasion",
    "shape", "joseki", "proverb", "sacrifice", "damezumari",
    "nakade", "bent four", "bulky five", "rabbity six",
    "false eye", "double atari", "under the stones",
})

GO_TECHNIQUE_PATTERN = re.compile(
    r"\b(" + "|".join(re.escape(t) for t in sorted(GO_TECHNIQUES, key=len, reverse=True)) + r")\b",
    re.IGNORECASE,
)
'''

_TEACHING_SCHEMA = r'''"""Colab stub: tagged text format parser for v3 SFT output."""
import re

_SECTION_RE = re.compile(r"^---(CORRECT|WRONG|HINT)---$", re.MULTILINE)


def parse_tagged_text(text: str) -> tuple[str, list[str], list[str]]:
    """Parse tagged text -> (correct_comment, wrong_comments, hints).

    Raises ValueError if no ---CORRECT--- section found.
    """
    tokens = _SECTION_RE.split(text)
    correct_comment = ""
    wrong_comments: list[str] = []
    hints: list[str] = []
    i = 0
    while i < len(tokens):
        token = tokens[i].strip()
        if token == "CORRECT" and i + 1 < len(tokens):
            correct_comment = tokens[i + 1].strip()
            i += 2
        elif token == "WRONG" and i + 1 < len(tokens):
            content = tokens[i + 1].strip()
            if content:
                wrong_comments.append(content)
            i += 2
        elif token == "HINT" and i + 1 < len(tokens):
            content = tokens[i + 1].strip()
            if content:
                hints.append(content)
            i += 2
        else:
            i += 1
    if not correct_comment:
        raise ValueError("No ---CORRECT--- section found in tagged text")
    return correct_comment, wrong_comments, hints
'''

_SCORERS = r'''"""Layer A (structural) and Layer B (grounded) scorers. Pure, deterministic, free.

Supports both tagged text (v3+) and JSON (backward compat) output formats.
"""
from __future__ import annotations

import json
import re
from dataclasses import asdict, dataclass

from tools.core.go_teaching_constants import GO_TECHNIQUE_PATTERN
from tools.core.teaching_schema import parse_tagged_text

_JSON_BLOCK_RE = re.compile(r"\{.*\}", re.DOTALL)
_SGF_COORD_RE = re.compile(r"\b[a-s]{2}\b")


@dataclass
class StructuralScore:
    parsed_ok: bool
    has_correct: bool
    has_wrong: bool
    n_hints: int
    n_chars_correct: int
    n_chars_wrong: int


@dataclass
class GroundedScore:
    mentions_correct_move: bool
    mentions_any_tag_technique: bool
    no_off_board_coords: bool
    looks_english: bool
    techniques_matched: list[str]
    correct_move_coord: str


def _score_structural_tagged(generated: str) -> StructuralScore:
    """Score tagged text format (---CORRECT---/---WRONG---/---HINT---)."""
    try:
        correct, wrongs, hints = parse_tagged_text(generated)
    except ValueError:
        return StructuralScore(False, False, False, 0, 0, 0)

    wc_text = " ".join(wrongs)
    return StructuralScore(
        parsed_ok=True,
        has_correct=bool(correct.strip()),
        has_wrong=bool(wc_text.strip()),
        n_hints=len(hints),
        n_chars_correct=len(correct),
        n_chars_wrong=len(wc_text),
    )


def _score_structural_json(generated: str) -> StructuralScore | None:
    """Try to score as JSON format. Returns None if not JSON."""
    blob = _JSON_BLOCK_RE.search(generated or "")
    if not blob:
        return None
    try:
        obj = json.loads(blob.group(0))
    except json.JSONDecodeError:
        return None

    tc = obj.get("teaching_comments") or {}
    cc = (tc.get("correct_comment") or "").strip() if isinstance(tc, dict) else ""
    wc = tc.get("wrong_comments") if isinstance(tc, dict) else None
    if isinstance(wc, dict):
        wc_text = " ".join(str(v) for v in wc.values())
    elif isinstance(wc, list):
        wc_text = " ".join(str(v) for v in wc)
    else:
        wc_text = ""
    hints = obj.get("hints") or []
    return StructuralScore(
        parsed_ok=True,
        has_correct=bool(cc),
        has_wrong=bool(wc_text.strip()),
        n_hints=len(hints) if isinstance(hints, list) else 0,
        n_chars_correct=len(cc),
        n_chars_wrong=len(wc_text),
    )


def score_structural(generated: str) -> StructuralScore:
    """Layer A: did the model emit a parseable response with the right shape?

    Supports both JSON (backward compat) and tagged text (v3+) formats.
    Tries JSON first; falls back to tagged text.
    """
    json_result = _score_structural_json(generated)
    if json_result is not None:
        return json_result
    return _score_structural_tagged(generated)


def _flatten_text(parsed_obj: dict) -> str:
    out: list[str] = []

    def walk(x):
        if isinstance(x, str):
            out.append(x)
        elif isinstance(x, dict):
            for v in x.values():
                walk(v)
        elif isinstance(x, list):
            for v in x:
                walk(v)

    walk(parsed_obj)
    return " ".join(out)


def score_grounded(
    generated: str,
    correct_move_coord: str,
    puzzle_tags: list[str],
    setup_coords: list[str],
) -> GroundedScore:
    blob = _JSON_BLOCK_RE.search(generated or "")
    parsed: dict = {}
    if blob:
        try:
            parsed = json.loads(blob.group(0))
        except json.JSONDecodeError:
            parsed = {}
    text = _flatten_text(parsed) if parsed else (generated or "")
    text_lower = text.lower()

    mentions_correct = bool(correct_move_coord) and (correct_move_coord.lower() in text_lower)

    techniques_matched: list[str] = []
    tag_tokens: set[str] = set()
    for t in puzzle_tags:
        for piece in re.split(r"[-_,\s]+", t.lower()):
            if piece:
                tag_tokens.add(piece)
    for m in GO_TECHNIQUE_PATTERN.finditer(text_lower):
        word = m.group(1).lower()
        if word in tag_tokens and word not in techniques_matched:
            techniques_matched.append(word)
    mentions_tag_tech = bool(techniques_matched) or not tag_tokens

    coord_set = {c.lower() for c in setup_coords if isinstance(c, str)}
    if correct_move_coord:
        coord_set.add(correct_move_coord.lower())
    found_coords = _SGF_COORD_RE.findall(text_lower)
    unknown = [c for c in found_coords if c not in coord_set]
    no_off_board = (len(unknown) <= max(1, len(found_coords) // 3))

    letters = sum(1 for ch in text if "a" <= ch.lower() <= "z")
    looks_english = (letters / max(len(text), 1)) >= 0.6 if text else False

    return GroundedScore(
        mentions_correct_move=mentions_correct,
        mentions_any_tag_technique=mentions_tag_tech,
        no_off_board_coords=no_off_board,
        looks_english=looks_english,
        techniques_matched=techniques_matched,
        correct_move_coord=correct_move_coord,
    )


def to_dict(obj) -> dict:
    return asdict(obj)
'''

_JUDGES = r'''"""Layer C judges. Notebook depends only on Judge protocol; pick a backend at runtime."""
from __future__ import annotations

import json
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Protocol


@dataclass
class JudgeResult:
    score: float
    rubric: dict
    rationale: str = ""
    backend: str = ""
    raw: dict = field(default_factory=dict)


class Judge(Protocol):
    def grade(
        self,
        prompt: str,
        generated: str,
        reference: str | None,
        metadata: dict,
    ) -> JudgeResult: ...

    def finalize(self) -> None: ...


class ManualJudge:
    def __init__(self, queue_path: Path | str, sample_n: int = 20):
        self.queue_path = Path(queue_path)
        self.sample_n = sample_n
        self._buffer: list[dict] = []

    def grade(
        self,
        prompt: str,
        generated: str,
        reference: str | None,
        metadata: dict,
    ) -> JudgeResult:
        self._buffer.append({
            "prompt": prompt,
            "generated": generated,
            "reference": reference,
            "metadata": metadata,
            "score": None,
            "rationale": "",
        })
        return JudgeResult(score=-1.0, rubric={}, rationale="awaiting human review", backend="manual")

    def finalize(self) -> None:
        self.queue_path.parent.mkdir(parents=True, exist_ok=True)
        items = self._buffer
        if len(items) > self.sample_n:
            import random
            random.seed(42)
            items = random.sample(items, self.sample_n)
        with self.queue_path.open("w", encoding="utf-8") as f:
            for it in items:
                f.write(json.dumps(it, ensure_ascii=False) + "\n")


def to_dict(r: JudgeResult) -> dict:
    return asdict(r)
'''

_RUNNER = r'''"""Inference + scoring loop. Notebook entry point: evaluate(...)."""
from __future__ import annotations

import json
import re
import time
from pathlib import Path
from typing import Optional

from .judges import Judge
from .scorers import GroundedScore, StructuralScore, score_grounded, score_structural, to_dict

_USER_BOARD_RE = re.compile(r"Board: (\d+)x\d+")
_USER_TAGS_RE = re.compile(r"^Tags: (.+)$", re.MULTILINE)
_USER_BLACK_RE = re.compile(r"^Black stones: (.+)$", re.MULTILINE)
_USER_WHITE_RE = re.compile(r"^White stones: (.+)$", re.MULTILINE)


def _parse_user_prompt(text: str) -> tuple[list[str], list[str], list[str]]:
    tags: list[str] = []
    black: list[str] = []
    white: list[str] = []
    if (m := _USER_TAGS_RE.search(text)):
        tags = [t.strip() for t in m.group(1).split(",") if t.strip()]
    if (m := _USER_BLACK_RE.search(text)):
        black = [c.strip() for c in m.group(1).split(",") if c.strip()]
    if (m := _USER_WHITE_RE.search(text)):
        white = [c.strip() for c in m.group(1).split(",") if c.strip()]
    return tags, black, white


def _correct_move_from_reference(reference_assistant: str | None) -> str:
    if not reference_assistant:
        return ""
    m = re.search(r"\{!([a-s]{2})\}", reference_assistant)
    return m.group(1) if m else ""


def generate_one(model, tokenizer, messages: list[dict], max_new_tokens: int = 384) -> str:
    import torch
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


def evaluate(
    model,
    tokenizer,
    test_rows: list[dict],
    out_dir: str | Path,
    judge: Optional[Judge] = None,
    max_new_tokens: int = 384,
    log_every: int = 25,
    sidecars: list[dict] | None = None,
    test_set_id: str = "",
) -> dict:
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    n = len(test_rows)
    results: list[dict] = []
    t0 = time.time()

    print(f"[eval] running on {n} test rows")
    for i, row in enumerate(test_rows):
        msgs = row["messages"]
        prompt_msgs = [m for m in msgs if m["role"] != "assistant"]
        reference = next((m["content"] for m in msgs if m["role"] == "assistant"), None)
        user_text = next((m["content"] for m in msgs if m["role"] == "user"), "")

        generated = generate_one(model, tokenizer, prompt_msgs, max_new_tokens=max_new_tokens)

        struct = score_structural(generated)

        tags, black, white = _parse_user_prompt(user_text)
        if sidecars is not None and i < len(sidecars):
            sc = sidecars[i] or {}
            correct_move = sc.get("correct_first_move") or _correct_move_from_reference(reference)
            sc_tags = sc.get("tags") or sc.get("techniques_found") or []
            if sc_tags and not tags:
                tags = list(sc_tags)
        else:
            correct_move = _correct_move_from_reference(reference)
        ground = score_grounded(
            generated,
            correct_move_coord=correct_move,
            puzzle_tags=tags,
            setup_coords=black + white,
        )

        judge_result = None
        if judge is not None:
            jr = judge.grade(
                prompt=user_text,
                generated=generated,
                reference=reference,
                metadata={"tags": tags, "correct_move_coord": correct_move},
            )
            judge_result = to_dict(jr)

        results.append({
            "idx": i,
            "structural": to_dict(struct),
            "grounded": to_dict(ground),
            "judge": judge_result,
            "generated_preview": generated[:600],
            "reference_preview": (reference or "")[:600],
        })

        if (i + 1) % log_every == 0:
            print(f"  {i + 1}/{n} ({(time.time() - t0) / 60:.1f} min elapsed)")

    if judge is not None:
        judge.finalize()

    elapsed_min = (time.time() - t0) / 60
    summary = _aggregate(results)
    summary["n"] = n
    summary["elapsed_minutes"] = round(elapsed_min, 2)

    (out_dir / "test_results.json").write_text(
        json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    (out_dir / "test_summary.json").write_text(
        json.dumps(summary, indent=2), encoding="utf-8"
    )
    _print_summary(summary)
    return summary


def _aggregate(results: list[dict]) -> dict:
    n = max(len(results), 1)
    s = [r["structural"] for r in results]
    g = [r["grounded"] for r in results]

    pct = lambda flag_list: round(100.0 * sum(1 for x in flag_list if x) / n, 1)

    summary = {
        "structural": {
            "format_compliance_pct": pct([x["parsed_ok"] for x in s]),
            "has_correct_pct": pct([x["has_correct"] for x in s]),
            "has_wrong_pct": pct([x["has_wrong"] for x in s]),
            "avg_hints": round(sum(x["n_hints"] for x in s) / n, 2),
            "avg_chars_correct": round(sum(x["n_chars_correct"] for x in s) / n, 1),
            "avg_chars_wrong": round(sum(x["n_chars_wrong"] for x in s) / n, 1),
        },
        "grounded": {
            "mentions_correct_move_pct": pct([x["mentions_correct_move"] for x in g]),
            "mentions_tag_technique_pct": pct([x["mentions_any_tag_technique"] for x in g]),
            "no_off_board_coords_pct": pct([x["no_off_board_coords"] for x in g]),
            "looks_english_pct": pct([x["looks_english"] for x in g]),
        },
    }

    useful = sum(
        1 for r in results
        if r["structural"]["parsed_ok"]
        and r["structural"]["has_correct"]
        and (r["grounded"]["mentions_correct_move"] or r["grounded"]["mentions_any_tag_technique"])
        and r["grounded"]["looks_english"]
    )
    summary["useful_answer_pct"] = round(100.0 * useful / n, 1)
    return summary


def evaluate_test_sets(
    model,
    tokenizer,
    test_sets: dict,
    out_dir: str | Path,
    judge: Optional[Judge] = None,
    max_new_tokens: int = 384,
    log_every: int = 25,
) -> dict:
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    per_set: dict[str, dict] = {}
    for ts_id, bundle in test_sets.items():
        chat_rows = bundle.get("chat_rows") or []
        sidecars = bundle.get("sidecars")
        if not chat_rows:
            print(f"[eval] skipping {ts_id}: no rows")
            continue
        print(f"\n[eval] === test set: {ts_id} ===")
        per_set[ts_id] = evaluate(
            model, tokenizer, chat_rows, out_dir / ts_id,
            judge=judge, max_new_tokens=max_new_tokens, log_every=log_every,
            sidecars=sidecars, test_set_id=ts_id,
        )

    cmp = {
        ts_id: {
            "n": s.get("n"),
            "useful_answer_pct": s.get("useful_answer_pct"),
            "format_compliance_pct": s.get("structural", {}).get("format_compliance_pct"),
            "mentions_correct_move_pct": s.get("grounded", {}).get("mentions_correct_move_pct"),
            "mentions_tag_technique_pct": s.get("grounded", {}).get("mentions_tag_technique_pct"),
            "no_off_board_coords_pct": s.get("grounded", {}).get("no_off_board_coords_pct"),
            "looks_english_pct": s.get("grounded", {}).get("looks_english_pct"),
        }
        for ts_id, s in per_set.items()
    }
    (out_dir / "comparison.json").write_text(
        json.dumps(cmp, indent=2), encoding="utf-8",
    )
    print("\n=== TEST-SET COMPARISON ===")
    cols = ["n", "useful_answer_pct", "format_compliance_pct",
            "mentions_correct_move_pct", "mentions_tag_technique_pct",
            "no_off_board_coords_pct", "looks_english_pct"]
    header = f"{'test_set':<32}" + "".join(f"{c:>12}" for c in cols)
    print(header)
    print("-" * len(header))
    for ts_id, row in cmp.items():
        line = f"{ts_id:<32}" + "".join(f"{str(row.get(c, '')):>12}" for c in cols)
        print(line)
    return {"per_set": per_set, "comparison": cmp}


def load_test_set_bundle(refined_dir: str | Path, test_set_id: str) -> dict:
    refined_dir = Path(refined_dir)
    chat_path = refined_dir / f"test_{test_set_id}.jsonl"
    meta_path = refined_dir / f"test_{test_set_id}_metadata.jsonl"
    chat_rows: list[dict] = []
    sidecars: list[dict] = []
    if chat_path.exists():
        with chat_path.open("r", encoding="utf-8") as f:
            for line in f:
                chat_rows.append(json.loads(line))
    if meta_path.exists():
        with meta_path.open("r", encoding="utf-8") as f:
            for line in f:
                sidecars.append(json.loads(line))
    return {"chat_rows": chat_rows, "sidecars": sidecars or None}


def _print_summary(summary: dict) -> None:
    print("\n=== EVAL SUMMARY ===")
    print(f"  test rows:         {summary.get('n')}")
    print(f"  elapsed:           {summary.get('elapsed_minutes')} min")
    print(f"  USEFUL ANSWER %:   {summary['useful_answer_pct']}    <-- single headline metric")
    print("  -- Structural (Layer A) --")
    for k, v in summary["structural"].items():
        print(f"    {k:<24} {v}")
    print("  -- Grounded (Layer B) --")
    for k, v in summary["grounded"].items():
        print(f"    {k:<28} {v}")
'''

_FILES = {
    "tools/__init__.py": "",
    "tools/core/__init__.py": "",
    "tools/core/go_teaching_constants.py": _GO_CONST,
    "tools/core/teaching_schema.py": _TEACHING_SCHEMA,
    "tools/yen_sei/__init__.py": "",
    "tools/yen_sei/eval/__init__.py": "",
    "tools/yen_sei/eval/scorers.py": _SCORERS,
    "tools/yen_sei/eval/judges.py": _JUDGES,
    "tools/yen_sei/eval/runner.py": _RUNNER,
}
for _rel, _content in _FILES.items():
    _p = BUNDLE_ROOT / _rel
    _p.parent.mkdir(parents=True, exist_ok=True)
    _p.write_text(_content, encoding="utf-8")

if str(BUNDLE_ROOT) not in sys.path:
    sys.path.insert(0, str(BUNDLE_ROOT))

from tools.yen_sei.eval.runner import evaluate_test_sets  # noqa: F401 sanity check
print(f"yen_sei eval bundle ready at {BUNDLE_ROOT}")

In [ ]:
# =====================================================================
# Cell 2 — CONFIG. Edit MODEL_NAME if 02a picked Gemma instead of Phi.
# =====================================================================

# Default winner; override after running 02a.
MODEL_NAME  = "microsoft/Phi-4-mini-instruct"

# Hyperparameters from PLAN.md
MAX_SEQ_LEN  = 2048
EPOCHS       = 3
BATCH_SIZE   = 4
GRAD_ACCUM   = 4         # effective batch = 16
LR           = 2e-4
LORA_R       = 32
LORA_ALPHA   = 64
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10

# Limits (set to None to use all). Useful for quick smoke runs.
# v2.3 row counts: train=2855, val=316, test_*={200,150,100,100}.
TRAIN_LIMIT  = None
VAL_LIMIT    = None
TEST_LIMIT   = None      # applied per test set independently

# Output / save behaviour
OUT_DIR             = "/content/yen_sei_tier1"
SAVE_MERGED_FP16    = True   # merge LoRA into base + save fp16 (large; download as zip)
PUSH_TO_HF          = False  # set True + add HF_TOKEN below to push adapter to your hub
HF_REPO_ID          = ""     # e.g. "your-username/yen-sei-tier1-phi4-lora"
HF_TOKEN            = ""     # paste a write token if PUSH_TO_HF=True

os.makedirs(OUT_DIR, exist_ok=True)
print(f"MODEL_NAME = {MODEL_NAME}")
print(f"OUT_DIR    = {OUT_DIR}")


## Upload data

The next cell looks for `train.jsonl`, `val.jsonl` in `/content/`. If missing, it opens an upload widget — drop all files from `tools/yen_sei/data/refined/` (train, val, and any `test_*.jsonl` + sidecars).

In [ ]:
# =====================================================================
# Cell 3 — Locate train/val + named TEST SETS.
#
# Schema v2.3+: train.jsonl + val.jsonl come from the curated training pool
# (gold + silver + filtered+capped bronze). The "test" split inside the
# training pool is empty by design (split_ratios.test=0.0). Real test sets
# are emitted by `tools.yen_sei eval-prep` as test_{id}.jsonl + sidecar
# test_{id}_metadata.jsonl, drawn from the marker-only pool (no prose to
# leak) and a small held-out gold/silver sanity set.
# =====================================================================
import json
from pathlib import Path

REQUIRED = ["train.jsonl", "val.jsonl"]
SEARCH_DIRS = [Path("/content"), Path.cwd(), Path("/content/drive/MyDrive/yen_sei")]

def find_data():
    found = {}
    for name in REQUIRED:
        for d in SEARCH_DIRS:
            p = d / name
            if p.exists():
                found[name] = p
                break
    return found

found = find_data()
missing = [n for n in REQUIRED if n not in found]
if missing:
    print(f"Missing: {missing}. Opening upload widget…")
    try:
        from google.colab import files  # type: ignore
        uploaded = files.upload()
        for name in uploaded:
            dest = Path("/content") / name
            print(f"  saved {name} -> {dest} ({len(uploaded[name])} bytes)")
        found = find_data()
    except ImportError:
        raise SystemExit(
            "Not in Colab. Place train.jsonl, val.jsonl + test_*.jsonl "
            "next to this notebook or in /content/."
        )

assert all(n in found for n in REQUIRED), f"Still missing: {set(REQUIRED) - set(found)}"
TRAIN_PATH = found["train.jsonl"]
VAL_PATH   = found["val.jsonl"]
DATA_DIR   = TRAIN_PATH.parent
print(f"\ntrain: {TRAIN_PATH}\nval:   {VAL_PATH}")

def load_jsonl(path, limit=None):
    rows = []
    with open(path, "r", encoding="utf-8-sig") as f:
        for line in f:
            rows.append(json.loads(line))
            if limit and len(rows) >= limit:
                break
    return rows

train_rows = load_jsonl(TRAIN_PATH, limit=TRAIN_LIMIT)
val_rows   = load_jsonl(VAL_PATH,   limit=VAL_LIMIT)
print(f"\nLoaded train={len(train_rows)}, val={len(val_rows)}.")

# Discover all test_*.jsonl in DATA_DIR (excluding *_metadata.jsonl)
test_set_files = sorted(
    p for p in DATA_DIR.glob("test_*.jsonl")
    if not p.name.endswith("_metadata.jsonl")
)
TEST_SETS: dict[str, dict] = {}
for chat_path in test_set_files:
    ts_id = chat_path.stem.removeprefix("test_")
    meta_path = DATA_DIR / f"test_{ts_id}_metadata.jsonl"
    chat_rows = load_jsonl(chat_path, limit=TEST_LIMIT)
    sidecars = load_jsonl(meta_path, limit=TEST_LIMIT) if meta_path.exists() else None
    TEST_SETS[ts_id] = {"chat_rows": chat_rows, "sidecars": sidecars}
    print(f"  test set '{ts_id}': {len(chat_rows)} rows"
          f"{' (with sidecar)' if sidecars else ''}")

if not TEST_SETS:
    print("\nWARNING: no test_*.jsonl found in", DATA_DIR)
    print("Run `python -m tools.yen_sei eval-prep` to generate them.")


## Train (QLoRA SFT)

Loads the model in 4-bit, attaches LoRA adapters, runs SFT for `EPOCHS` epochs with cosine LR schedule and warmup. Saves the best checkpoint by val loss. ~3-6 hours on T4 for the full dataset.


In [ ]:
# =====================================================================
# Cell 4 — Load model + LoRA adapter, prepare dataset, train.
# =====================================================================
import torch
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig

print(f"Loading {MODEL_NAME} in 4-bit…")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    dtype=None,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules="all-linear",
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

def to_text(example):
    return {"text": tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False
    )}

train_ds = Dataset.from_list(train_rows).map(to_text, remove_columns=["messages"])
val_ds   = Dataset.from_list(val_rows).map(to_text,   remove_columns=["messages"])

steps_per_epoch = max(1, len(train_ds) // (BATCH_SIZE * GRAD_ACCUM))
total_steps     = steps_per_epoch * EPOCHS
warmup_steps    = max(1, int(total_steps * WARMUP_RATIO))
print(f"\nTotal steps: {total_steps}  (steps/epoch={steps_per_epoch}, warmup={warmup_steps})")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    args=SFTConfig(
        output_dir=os.path.join(OUT_DIR, "checkpoints"),
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR,
        lr_scheduler_type="cosine",
        warmup_steps=warmup_steps,
        weight_decay=WEIGHT_DECAY,
        logging_steps=20,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        bf16=torch.cuda.is_bf16_supported(),
        fp16=not torch.cuda.is_bf16_supported(),
        optim="adamw_8bit",
        report_to="none",
        seed=42,
    ),
)

t0 = time.time()
trainer.train()
train_secs = time.time() - t0
peak_vram = torch.cuda.max_memory_allocated() / (1024 ** 3) if torch.cuda.is_available() else 0
print(f"\nTraining done in {train_secs/60:.1f} min, peak VRAM: {peak_vram:.1f} GB")


## Evaluate on named test sets

In [ ]:
# =====================================================================
# Cell 5 — Eval on every named test set + comparison table.
#
# Three layers (handled inside tools.yen_sei.eval.runner):
#   A. Structural — valid tagged text format with the right shape
#                   (---CORRECT--- / ---WRONG--- / ---HINT--- delimiters).
#                   Also accepts JSON for backward compat with pre-v3 models.
#   B. Grounded   — output mentions the actual correct move + a known
#                   technique tag for THIS position. For marker-only test
#                   sets the correct first-move SGF coord is read from
#                   the sidecar (sidecars[i].correct_first_move) instead
#                   of being parsed out of the reference assistant message.
#   C. ManualJudge — 20 random samples written to judge_queue.jsonl for
#                    eyeball review. (Optional pluggable Judge supported
#                    by the runner; not enabled here.)
# Headline metric per test set is "useful_answer_pct".
# =====================================================================
import json, time, random, os
from pathlib import Path
from unsloth import FastLanguageModel  # type: ignore
from tools.yen_sei.eval.runner import evaluate_test_sets

FastLanguageModel.for_inference(model)

t0 = time.time()
eval_out = evaluate_test_sets(
    model=model,
    tokenizer=tokenizer,
    test_sets=TEST_SETS,                       # populated in Cell 3
    out_dir=Path(OUT_DIR) / "eval",
    judge=None,                                 # plug a Judge here for Layer C
    max_new_tokens=384,
    log_every=25,
)
elapsed_min = (time.time() - t0) / 60

# evaluate_test_sets returns {"per_set": {ts_id: summary}, "comparison": {...}}
print(f"\nTotal eval time: {elapsed_min:.1f} min across {len(TEST_SETS)} test sets.")

# Persist a top-level run summary alongside the per-set artefacts.
run_summary = {
    "model": MODEL_NAME,
    "train_seconds": train_secs,
    "peak_vram_gb": peak_vram,
    "eval_minutes": round(elapsed_min, 2),
    "comparison": eval_out["comparison"],
}
summary_path = Path(OUT_DIR) / "eval" / "run_summary.json"
summary_path.parent.mkdir(parents=True, exist_ok=True)
summary_path.write_text(json.dumps(run_summary, indent=2), encoding="utf-8")
print(f"\nRun summary -> {summary_path}")

# Layer C — sample 20 random rows across all test sets for manual review.
random.seed(42)
manual_pool = []
for ts_id, bundle in TEST_SETS.items():
    rows = bundle["chat_rows"]
    sidecars = bundle.get("sidecars") or [None] * len(rows)
    per_set_results = json.loads(
        (Path(OUT_DIR) / "eval" / ts_id / "test_results.json").read_text(encoding="utf-8")
    )
    for chat, side, res in zip(rows, sidecars, per_set_results):
        user_text = next((m["content"] for m in chat["messages"] if m["role"] == "user"), "")
        manual_pool.append({
            "test_set_id": ts_id,
            "user": user_text,
            "correct_first_move": (side or {}).get("correct_first_move"),
            "generated": res.get("generated_preview", ""),
            "score": None, "rationale": "",
        })
manual_sample = random.sample(manual_pool, min(20, len(manual_pool)))
queue_path = Path(OUT_DIR) / "judge_queue.jsonl"
with queue_path.open("w", encoding="utf-8") as f:
    for it in manual_sample:
        f.write(json.dumps(it, ensure_ascii=False) + "\n")
print(f"\n[Layer C] {len(manual_sample)} samples queued for manual review -> {queue_path}")
print("           Open the file, fill in `score` (0..5) and `rationale`, save.")

## Save adapter (+ optional merged fp16) and download


In [ ]:
# =====================================================================
# Cell 6 - Save LoRA adapter, optional merged fp16, zip, download.
# =====================================================================
import shutil

adapter_dir = os.path.join(OUT_DIR, "adapter")
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print(f"Saved LoRA adapter -> {adapter_dir}")

merged_dir = None
if SAVE_MERGED_FP16:
    merged_dir = os.path.join(OUT_DIR, "merged_fp16")
    try:
        model.save_pretrained_merged(merged_dir, tokenizer, save_method="merged_16bit")
        print(f"Saved merged fp16 weights -> {merged_dir}")
    except Exception as e:
        print(f"WARNING: merged fp16 save failed ({type(e).__name__}: {e})")
        print("  Adapter-only save above is still usable; merge offline if needed.")
        merged_dir = None

# Zip everything for one-click download
artifact_root = os.path.join(OUT_DIR, "artifacts")
os.makedirs(artifact_root, exist_ok=True)

# Copy the eval/ tree as a whole — preserves per-test-set subdirs +
# comparison.json + run_summary.json from Cell 5.
eval_src = os.path.join(OUT_DIR, "eval")
if os.path.exists(eval_src):
    shutil.copytree(eval_src, os.path.join(artifact_root, "eval"), dirs_exist_ok=True)
judge_queue = os.path.join(OUT_DIR, "judge_queue.jsonl")
if os.path.exists(judge_queue):
    shutil.copy(judge_queue, os.path.join(artifact_root, "judge_queue.jsonl"))
shutil.copytree(adapter_dir, os.path.join(artifact_root, "adapter"), dirs_exist_ok=True)
if merged_dir:
    shutil.copytree(merged_dir, os.path.join(artifact_root, "merged_fp16"), dirs_exist_ok=True)

zip_base = os.path.join(OUT_DIR, "yen_sei_tier1_artifacts")
zip_path = shutil.make_archive(zip_base, "zip", artifact_root)
size_mb = os.path.getsize(zip_path) / (1024 ** 2)
print(f"\nZipped artifacts -> {zip_path}  ({size_mb:.1f} MB)")

# Optional HF push
if PUSH_TO_HF and HF_REPO_ID and HF_TOKEN:
    try:
        from huggingface_hub import HfApi  # type: ignore
        api = HfApi(token=HF_TOKEN)
        api.create_repo(HF_REPO_ID, exist_ok=True, private=True)
        api.upload_folder(folder_path=adapter_dir, repo_id=HF_REPO_ID, path_in_repo="adapter")
        if merged_dir:
            api.upload_folder(folder_path=merged_dir, repo_id=HF_REPO_ID, path_in_repo="merged_fp16")
        print(f"Pushed to https://huggingface.co/{HF_REPO_ID}")
    except Exception as e:
        print(f"HF push failed ({type(e).__name__}: {e})")

# Browser download (Colab only). Skipped automatically if not in Colab.
try:
    from google.colab import files  # type: ignore
    files.download(zip_path)
except ImportError:
    print(f"\nNot in Colab. Artifacts at: {artifact_root}")
